# 🧠 LoRA Fine-Tuning Studio — Google Colab

Fine-tune any LLM using LoRA/QLoRA on the free Colab T4 GPU.

**Runtime → Change runtime type → T4 GPU** before running!

---
| Step | What happens |
|------|--------------|
| 1 | Install dependencies |
| 2 | Load base model (4-bit QLoRA) |
| 3 | Load & format dataset |
| 4 | Train LoRA adapters |
| 5 | Evaluate & chat with the model |
| 6 | Push adapters to HuggingFace Hub |

In [ ]:
# ── Step 0: Check GPU ─────────────────────────────────────────
!nvidia-smi
import torch
print(f'\nPyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

In [ ]:
# ── Step 1: Install dependencies ─────────────────────────────
# This takes ~2 minutes on first run
!pip install -q \
    transformers>=4.40.0 \
    peft>=0.10.0 \
    trl>=0.8.0 \
    datasets>=2.18.0 \
    accelerate>=0.29.0 \
    bitsandbytes>=0.43.0 \
    huggingface_hub \
    rouge-score \
    rich

print('✓ All dependencies installed')

In [ ]:
# ── Step 2: Configuration ─────────────────────────────────────
# Edit these values to customize your training run

BASE_MODEL   = 'TinyLlama/TinyLlama-1.1B-Chat-v1.0'  # Base model to fine-tune
DATASET_NAME = 'yahma/alpaca-cleaned'                  # HF dataset ID
OUTPUT_DIR   = '/content/lora-output'                  # Where to save checkpoints

# LoRA hyperparameters
LORA_R       = 16    # Rank
LORA_ALPHA   = 32    # Alpha (2x rank is a good default)
LORA_DROPOUT = 0.05

# Training
NUM_EPOCHS   = 1     # 1 epoch ≈ 30 min on T4 with full alpaca; set 3 for best quality
BATCH_SIZE   = 4
GRAD_ACCUM   = 4     # Effective batch = BATCH_SIZE × GRAD_ACCUM = 16
LR           = 2e-4
MAX_LENGTH   = 512
SAMPLE_FRAC  = 0.1   # Use 10% of dataset for speed; set 1.0 for full training

# HuggingFace Hub (optional — fill in to push your model)
HF_USERNAME  = ''    # Your HF username
MODEL_REPO   = f'{HF_USERNAME}/tinyllama-alpaca-lora' if HF_USERNAME else ''

print('Configuration set!')
print(f'  Model: {BASE_MODEL}')
print(f'  Dataset: {DATASET_NAME} ({int(SAMPLE_FRAC*100)}% sample)')
print(f'  LoRA: r={LORA_R}, alpha={LORA_ALPHA}')
print(f'  Training: {NUM_EPOCHS} epoch(s), lr={LR}')

In [ ]:
# ── Step 3: Load Model (4-bit QLoRA) ─────────────────────────
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, TaskType, get_peft_model, prepare_model_for_kbit_training

# 4-bit quantization config
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
)

print(f'Loading {BASE_MODEL}...')
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map='auto',
    trust_remote_code=True,
)
model = prepare_model_for_kbit_training(model)

# Apply LoRA
peft_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    target_modules=['q_proj', 'v_proj', 'k_proj', 'o_proj'],
    bias='none',
)
model = get_peft_model(model, peft_config)

trainable, total = 0, 0
for _, p in model.named_parameters():
    total += p.numel()
    if p.requires_grad: trainable += p.numel()

print(f'\n✓ Model loaded with LoRA adapters')
print(f'  Trainable params: {trainable:,} ({100*trainable/total:.4f}% of {total:,} total)')

In [ ]:
# ── Step 4: Load & Format Dataset ────────────────────────────
from datasets import load_dataset

ALPACA_TEMPLATE = '''### Instruction:
{instruction}

### Input:
{input}

### Response:
{output}'''
ALPACA_TEMPLATE_NO_INPUT = '''### Instruction:
{instruction}

### Response:
{output}'''

def format_prompt(example):
    inp = example.get('input', '') or ''
    if inp.strip():
        text = ALPACA_TEMPLATE.format(
            instruction=example['instruction'],
            input=inp,
            output=example['output'],
        )
    else:
        text = ALPACA_TEMPLATE_NO_INPUT.format(
            instruction=example['instruction'],
            output=example['output'],
        )
    return {'text': text}

print(f'Loading dataset: {DATASET_NAME}')
ds = load_dataset(DATASET_NAME, split='train')

if SAMPLE_FRAC < 1.0:
    n = int(len(ds) * SAMPLE_FRAC)
    ds = ds.select(range(n))
    print(f'  Sampled {n} examples ({SAMPLE_FRAC*100:.0f}%)')

ds = ds.map(format_prompt, desc='Formatting prompts')
splits = ds.train_test_split(test_size=0.05, seed=42)

print(f'  Train: {len(splits["train"])} | Val: {len(splits["test"])}')
print('\nSample prompt:')
print(splits['train'][0]['text'][:300])

In [ ]:
# ── Step 5: Train! ────────────────────────────────────────────
import os
from transformers import TrainingArguments
from trl import SFTTrainer

os.makedirs(OUTPUT_DIR, exist_ok=True)

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LR,
    lr_scheduler_type='cosine',
    warmup_ratio=0.03,
    weight_decay=0.001,
    optim='paged_adamw_32bit',
    fp16=False,
    bf16=True,
    gradient_checkpointing=True,
    logging_steps=10,
    save_steps=100,
    evaluation_strategy='steps',
    eval_steps=100,
    load_best_model_at_end=True,
    report_to='none',
    seed=42,
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=splits['train'],
    eval_dataset=splits['test'],
    tokenizer=tokenizer,
    peft_config=peft_config,
    dataset_text_field='text',
    max_seq_length=MAX_LENGTH,
    packing=False,
)

print('Starting training...')
print(f'Estimated time: ~{NUM_EPOCHS * int(len(splits["train"]) * SAMPLE_FRAC / (BATCH_SIZE * GRAD_ACCUM)) // 60 + 5} minutes on T4')
trainer.train()

# Save final checkpoint
final_path = f'{OUTPUT_DIR}/checkpoint-final'
trainer.save_model(final_path)
tokenizer.save_pretrained(final_path)
print(f'\n✓ Training complete! Saved to {final_path}')

In [ ]:
# ── Step 6: Test the fine-tuned model ────────────────────────
from peft import PeftModel

def generate(instruction, input_text='', max_new_tokens=200, temperature=0.7):
    if input_text.strip():
        prompt = f'### Instruction:\n{instruction}\n\n### Input:\n{input_text}\n\n### Response:\n'
    else:
        prompt = f'### Instruction:\n{instruction}\n\n### Response:\n'
    
    inputs = tokenizer(prompt, return_tensors='pt').to(model.device)
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            do_sample=temperature > 0,
            pad_token_id=tokenizer.eos_token_id,
        )
    new_tokens = out[0][inputs['input_ids'].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

# Test examples
tests = [
    ('What is LoRA fine-tuning?', ''),
    ('Summarize this text.', 'The sky is blue because of Rayleigh scattering of sunlight.'),
    ('Write a Python function to check if a number is prime.', ''),
]

for instruction, inp in tests:
    print(f'Instruction: {instruction}')
    if inp: print(f'Input: {inp}')
    response = generate(instruction, inp)
    print(f'Response: {response}')
    print('-' * 60)

In [ ]:
# ── Step 7: Push to HuggingFace Hub (optional) ───────────────
# Fill in HF_USERNAME above and run this cell to upload your model

if MODEL_REPO:
    from huggingface_hub import notebook_login
    notebook_login()  # Opens a widget to enter your HF token
    
    print(f'Pushing adapters to {MODEL_REPO}...')
    model.push_to_hub(MODEL_REPO, private=False)
    tokenizer.push_to_hub(MODEL_REPO, private=False)
    print(f'✓ Done! View at: https://huggingface.co/{MODEL_REPO}')
else:
    print('Set HF_USERNAME in the config cell to push to Hub.')
    print('Alternatively, download the checkpoint from the Files panel (left sidebar).')
    print(f'Checkpoint is at: {final_path}')

In [ ]:
# ── Optional: Download checkpoint as zip ─────────────────────
import shutil
zip_path = '/content/lora-checkpoint'
shutil.make_archive(zip_path, 'zip', final_path)
print(f'Zip created: {zip_path}.zip')
print('Download it from the Files panel on the left (folder icon).')